# Moduł 15 — Strategie zawodów: 2. etap III OAI

Ten moduł to **taktyczny przewodnik** na 2. etap Olimpiady AI (13–15 marca 2026).  
Zawiera sprawdzone strategie, checklisty i wzorce, które pomogą Ci zmaksymalizować wynik.

## Format 2. etapu
| Element | Szczegóły |
|---|---|
| Czas | ~5h (ograniczony) |
| Środowisko | Jupyter Notebook |
| Internet | ❌ BRAK |
| GPU | T4 (16 GB) lub A100 (40/80 GB) |
| Biblioteki | torch, torchvision, numpy, pandas, sklearn, matplotlib, nltk, transformers, xgboost, tqdm, Pillow |
| LLM | Bielik-11B-v3.0-Instruct (lokalnie) |
| Ocenianie | Automatyczne (`FINAL_EVALUATION_MODE`) |
| Lokalizacje | Kraków, Poznań, Warszawa, Wrocław |

## 1. Wzorzec FINAL_EVALUATION_MODE — dryl

**Najważniejszy wzorzec na olimpiadzie!** Każde zadanie ma flagę, która przełącza między  
trybem ćwiczeniowym (małe dane) a oceną (pełne dane). Musisz to obsłużyć w KAŻDYM zadaniu.

In [ ]:
import numpy as np
import time

# ============================
# WZORZEC FINAL_EVALUATION_MODE
# ============================

# Ta flaga jest ustawiana przez system oceniający
FINAL_EVALUATION_MODE = False

if FINAL_EVALUATION_MODE:
    # Tryb oceny — pełne dane, pełny czas
    DATA_PATH = "/data/full/"         # Ścieżka do pełnych danych
    N_EPOCHS = 50                      # Więcej epok
    BATCH_SIZE = 64                    # Większy batch
    N_FOLDS = 5                        # Pełna walidacja
else:
    # Tryb deweloperski — małe dane, szybki feedback
    DATA_PATH = "./sample_data/"       # Dane przykładowe
    N_EPOCHS = 3                       # Szybko sprawdź czy działa
    BATCH_SIZE = 16                    # Mniejszy batch
    N_FOLDS = 2                        # Szybka walidacja

print(f"Tryb: {'OCENA' if FINAL_EVALUATION_MODE else 'DEWELOPERSKI'}")
print(f"Dane: {DATA_PATH}")
print(f"Epoki: {N_EPOCHS}, Batch: {BATCH_SIZE}, Folds: {N_FOLDS}")

# ============================
# WAŻNE ZASADY:
# ============================
print("\n📋 Zasady FINAL_EVALUATION_MODE:")
print("1. NIGDY nie hardkoduj ścieżek → użyj DATA_PATH")
print("2. Parametry muszą SKALOWAĆ SIĘ → więcej danych = więcej epok")
print("3. TESTUJ oba tryby zanim oddasz notebook")
print("4. Nie drukuj za dużo w trybie FINAL (spowalnia ewaluację)")
print("5. Wynik musi być zapisany do zmiennej/pliku wskazanego w zadaniu")

## 2. Zarządzanie czasem — strategia "Triage"

Na olimpiadzie masz ~5h na kilka zadań (prawdopodobnie 4-6). Nie próbuj rozwiązać wszystkich perfekcyjnie.  
Zastosuj **triage** — jak w medycynie ratunkowej.

In [ ]:
# === STRATEGIA TRIAGE ===

print("🏥 TRIAGE — Strategia podziału czasu")
print("=" * 60)

strategy = {
    "FAZA 1 — Rozpoznanie (15 min)": [
        "Przeczytaj WSZYSTKIE zadania",
        "Oceń trudność: Łatwe / Średnie / Trudne",
        "Oceń typ: ML / DL / NLP / CV / Inne",
        "Zaplanuj kolejność rozwiązywania",
    ],
    "FAZA 2 — Łatwe zadania (90 min)": [
        "Rozwiąż najpierw zadania, które WIESZ jak zrobić",
        "Cel: 80-100% punktów za te zadania",
        "Typowe: klasyczna ML, proste CNN, preprocessing",
    ],
    "FAZA 3 — Średnie zadania (120 min)": [
        "Zadania wymagające myślenia, ale znane techniki",
        "Cel: 60-80% punktów",
        "Tu może przydać się Bielik (NLP)",
    ],
    "FAZA 4 — Trudne zadania (60 min)": [
        "Zadania, które wymagają kreatywnego podejścia",
        "Cel: zdobyć JAKIEKOLWIEK punkty (nawet 30%)",
        "Lepsze częściowe rozwiązanie niż zero",
    ],
    "FAZA 5 — Weryfikacja (15 min)": [
        "Sprawdź FINAL_EVALUATION_MODE we WSZYSTKICH zadaniach",
        "Upewnij się, że wyniki są zapisywane poprawnie",
        "Przetestuj w obu trybach",
        "NIE zmieniaj nic w ostatniej chwili!",
    ],
}

for phase, actions in strategy.items():
    print(f"\n{phase}:")
    for action in actions:
        print(f"  • {action}")

print("\n" + "=" * 60)
print("⚠️ ZŁOTA ZASADA: Lepiej 4 zadania po 70% niż 2 zadania po 95%")
print("   Punkty się sumują — maximalizuj SUMĘ, nie perfekcję!")

## 3. Szybka ocena typu zadania

Na podstawie analizy 24 zadań z trzech edycji OAI — oto matryca typów:

In [ ]:
# === MATRYCA TYPÓW ZADAŃ ===

task_types = {
    "Klasyfikacja (obraz)": {
        "Częstość": "★★★★★ (najczęstsze)",
        "Techniki": "CNN, Transfer Learning, Data Augmentation",
        "Metryki": "Accuracy, F1, ROC AUC",
        "Czas": "30-60 min",
        "Trudność": "Łatwa-Średnia",
        "Pułapki": "Niezbalansowane klasy, multi-label",
    },
    "Klasyfikacja (tekst/tabela)": {
        "Częstość": "★★★★☆",
        "Techniki": "XGBoost, RF, SVM, Bielik",
        "Metryki": "Accuracy, F1, Balanced Accuracy",
        "Czas": "20-45 min",
        "Trudność": "Łatwa",
        "Pułapki": "Feature engineering, brakujące dane",
    },
    "Segmentacja/Detekcja": {
        "Częstość": "★★★☆☆",
        "Techniki": "U-Net, Conv+Deconv, maski",
        "Metryki": "IoU, Dice",
        "Czas": "45-90 min",
        "Trudność": "Średnia-Trudna",
        "Pułapki": "Wielokanałowe obrazy, augmentacja",
    },
    "NLP / Embeddingi": {
        "Częstość": "★★★☆☆",
        "Techniki": "Word2Vec, TF-IDF, Bielik, Procrustes",
        "Metryki": "Cosine similarity, BLEU, Balanced Acc",
        "Czas": "30-60 min",
        "Trudność": "Średnia",
        "Pułapki": "Polski tekst, tokenizacja",
    },
    "Autoenkodery / Generatywne": {
        "Częstość": "★★☆☆☆",
        "Techniki": "Conv AE, INR, denoising",
        "Metryki": "MSE, PSNR, SSIM",
        "Czas": "45-90 min",
        "Trudność": "Średnia-Trudna",
        "Pułapki": "Inpainting, rekonstrukcja",
    },
    "Adversarial / Bezpieczeństwo": {
        "Częstość": "★★☆☆☆",
        "Techniki": "FGSM, pruning, machine unlearning",
        "Metryki": "Accuracy drop, sparsity",
        "Czas": "30-60 min",
        "Trudność": "Trudna",
        "Pułapki": "Nieprzewidziane formaty, gradients",
    },
}

print("📊 MATRYCA TYPÓW ZADAŃ — na podstawie 24 zadań z I-III OAI")
print("=" * 65)

for task_type, info in task_types.items():
    print(f"\n🔹 {task_type}")
    for key, value in info.items():
        print(f"   {key}: {value}")

## 4. Szablony rozwiązań — kopiuj i wklej

Poniżej gotowe szablony, które możesz zapamiętać i dostosować na olimpiadzie.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================================
# SZABLON 1: Klasyfikacja obrazów (CNN)
# =========================================

class QuickCNN(nn.Module):
    """Szybki CNN do klasyfikacji — gotowy na olimpiadę."""
    def __init__(self, in_channels=3, num_classes=10, img_size=32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

# Test szablonu
model = QuickCNN(in_channels=3, num_classes=5, img_size=32)
x = torch.randn(4, 3, 32, 32)
out = model(x)
print(f"Szablon CNN: input {x.shape} → output {out.shape}")
print(f"Parametry: {sum(p.numel() for p in model.parameters()):,}")

print("\n" + "="*60)

# =========================================
# SZABLON 2: Pętla treningowa (uniwersalna)
# =========================================

def train_model(model, train_loader, val_loader, n_epochs, lr=1e-3, device='cpu'):
    """Uniwersalna pętla treningowa z early stopping."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0
    patience = 5
    no_improve = 0
    
    for epoch in range(n_epochs):
        # --- Trening ---
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # --- Walidacja ---
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                preds = model(X_batch).argmax(dim=1).cpu()
                all_preds.extend(preds.numpy())
                all_labels.extend(y_batch.numpy())
        
        val_acc = accuracy_score(all_labels, all_preds)
        
        if not FINAL_EVALUATION_MODE:
            print(f"Epoch {epoch+1}/{n_epochs} | Loss: {train_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve = 0
            # Zapisz najlepszy model
            best_state = model.state_dict().copy()
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping po {epoch+1} epokach")
                break
    
    model.load_state_dict(best_state)
    return model, best_val_acc

print("✅ Szablon pętli treningowej gotowy")
print("   Zawiera: CrossEntropyLoss, Adam, Early Stopping, Best Model")

In [ ]:
# =========================================
# SZABLON 3: Klasyfikacja tabelaryczna (XGBoost)
# =========================================

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Symulowane dane tabelaryczne
np.random.seed(42)
X = np.random.randn(500, 10)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

# Quick comparison — testuj 3 modele, wybierz najlepszy
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

try:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
except ImportError:
    print("XGBoost niedostępny — na olimpiadzie BĘDZIE dostępny")

print("=== Quick Model Comparison ===")
best_name, best_score = "", 0
for name, model in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    scores = cross_val_score(pipe, X, y, cv=5, scoring='f1_macro')
    mean_score = scores.mean()
    print(f"  {name:20s}: F1={mean_score:.4f} ± {scores.std():.4f}")
    if mean_score > best_score:
        best_name, best_score = name, mean_score

print(f"\n🏆 Najlepszy: {best_name} (F1={best_score:.4f})")
print("\n💡 Tip: Na olimpiadzie XGBoost jest prawie zawsze najlepszy na danych tabelarycznych")

print("\n" + "="*60)

# =========================================
# SZABLON 4: Metryki — gotowe funkcje
# =========================================

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, balanced_accuracy_score, classification_report,
    mean_squared_error, mean_absolute_error
)

def compute_all_metrics(y_true, y_pred, y_prob=None):
    """Oblicz wszystkie metryki naraz — oszczędność czasu."""
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro'),
    }
    if y_prob is not None and len(np.unique(y_true)) == 2:
        metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
    return metrics

# Test
y_true = np.array([0, 1, 1, 0, 1, 0, 1, 1])
y_pred = np.array([0, 1, 0, 0, 1, 1, 1, 1])

print("\n📏 Wszystkie metryki naraz:")
for metric, value in compute_all_metrics(y_true, y_pred).items():
    print(f"  {metric:25s}: {value:.4f}")

## 5. Debugowanie bez internetu

Na olimpiadzie **nie masz internetu**. Nie sprawdzisz StackOverflow.  
Musisz znać typowe błędy i ich rozwiązania.

In [ ]:
import torch
import numpy as np

print("🐛 NAJCZĘSTSZE BŁĘDY I ROZWIĄZANIA")
print("=" * 60)

bugs = [
    {
        "błąd": "RuntimeError: CUDA out of memory",
        "przyczyna": "Model/batch za duży dla GPU",
        "rozwiązania": [
            "Zmniejsz BATCH_SIZE (np. z 64 na 16)",
            "Użyj torch.cuda.empty_cache()",
            "Użyj with torch.no_grad(): (przy ewaluacji)",
            "Usuń niepotrzebne modele (del model; gc.collect())",
            "Użyj gradient accumulation",
        ]
    },
    {
        "błąd": "RuntimeError: mat1 and mat2 shapes cannot be multiplied",
        "przyczyna": "Niezgodność wymiarów w warstwie Linear",
        "rozwiązania": [
            "Sprawdź x.shape przed Linear",
            "Dodaj nn.Flatten() przed Linear",
            "Użyj AdaptiveAvgPool2d aby kontrolować rozmiar",
        ]
    },
    {
        "błąd": "ValueError: Target size (torch.Size([64])) must match input size (torch.Size([64, 10]))",
        "przyczyna": "Zły loss function lub format labeli",
        "rozwiązania": [
            "CrossEntropyLoss → y musi być (N,) z int",
            "BCELoss → y musi być (N, C) z float",
            "MSELoss → y musi mieć ten sam shape co output",
        ]
    },
    {
        "błąd": "Model accuracy stuck at ~random (25% for 4 classes)",
        "przyczyna": "Model się nie uczy",
        "rozwiązania": [
            "Sprawdź learning rate (za duży → NaN, za mały → stoi)",
            "Sprawdź czy optimizer.zero_grad() jest wywoływany",
            "Sprawdź czy loss.backward() jest wywoływany",
            "Normalizuj dane (StandardScaler / /255.0)",
            "Sprawdź czy labele zaczynają się od 0",
        ]
    },
    {
        "błąd": "FINAL_EVALUATION_MODE nie przechodzi",
        "przyczyna": "Ścieżki do danych, niepoprawny format wyników",
        "rozwiązania": [
            "Sprawdź czy ścieżki używają DATA_PATH",
            "Upewnij się, że wynik jest we właściwej zmiennej",
            "Sprawdź format (numpy array vs lista vs tensor)",
        ]
    },
]

for i, bug in enumerate(bugs, 1):
    print(f"\n{i}. {bug['błąd']}")
    print(f"   Przyczyna: {bug['przyczyna']}")
    print(f"   Rozwiązania:")
    for sol in bug['rozwiązania']:
        print(f"     → {sol}")

In [ ]:
# === NARZĘDZIA DEBUGOWANIA ===

import torch

def debug_tensor(name, tensor):
    """Szybka diagnoza tensora — skopiuj tę funkcję na olimpiadę!"""
    if isinstance(tensor, torch.Tensor):
        print(f"{name}: shape={tensor.shape}, dtype={tensor.dtype}, "
              f"device={tensor.device}, min={tensor.min():.4f}, max={tensor.max():.4f}, "
              f"mean={tensor.mean():.4f}, has_nan={tensor.isnan().any()}")
    elif isinstance(tensor, np.ndarray):
        print(f"{name}: shape={tensor.shape}, dtype={tensor.dtype}, "
              f"min={tensor.min():.4f}, max={tensor.max():.4f}, "
              f"mean={tensor.mean():.4f}, has_nan={np.isnan(tensor).any()}")
    else:
        print(f"{name}: type={type(tensor)}, value={tensor}")

# Przykład użycia
x = torch.randn(8, 3, 32, 32)
y = torch.randint(0, 5, (8,))

print("=== Debug tensors ===")
debug_tensor("input", x)
debug_tensor("labels", y)

# Sprawdzenie GPU
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print("\n💡 Skopiuj debug_tensor() na olimpiadę — oszczędzi mnóstwo czasu!")

## 6. Optymalizacja GPU — sztuczki

GPU na olimpiadzie to Twoja najcenniejsza zasoby. Używaj efektywnie.

In [ ]:
import torch
import torch.nn as nn
import time

# === SZTUCZKI GPU ===

print("⚡ OPTYMALIZACJA GPU")
print("=" * 60)

tips = [
    ("1. Automatyczny device", 
     'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")'),
    ("2. Mixed precision (szybciej + mniej VRAM)",
     'scaler = torch.cuda.amp.GradScaler()\nwith torch.cuda.amp.autocast(): ...'),
    ("3. Gradient accumulation (duży efektywny batch)",
     'if (i+1) % accum_steps == 0: optimizer.step(); optimizer.zero_grad()'),
    ("4. DataLoader z pin_memory",
     'DataLoader(..., pin_memory=True, num_workers=2)'),
    ("5. torch.no_grad() przy ewaluacji",
     'with torch.no_grad(): preds = model(x)'),
    ("6. torch.compile() (PyTorch 2.0+)",
     'model = torch.compile(model)  # Na A100 daje 2-3x speedup'),
]

for title, code in tips:
    print(f"\n  {title}")
    print(f"    >>> {code}")

# === Mixed Precision Training — wzorzec ===
print("\n" + "=" * 60)
print("\n📝 PEŁNY WZORZEC — Mixed Precision Training:")
print("""
scaler = torch.cuda.amp.GradScaler()

for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass w mixed precision
        with torch.cuda.amp.autocast():
            output = model(X_batch)
            loss = criterion(output, y_batch)
        
        # Backward pass z GradScaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
""")
print("💡 Mixed precision daje ~2x speedup i ~50% mniej VRAM!")

## 7. Checklist przed oddaniem

### ✅ Przed kliknięciem "Oddaj":

1. [ ] `FINAL_EVALUATION_MODE = True` → notebook działa poprawnie?
2. [ ] Ścieżki do danych używają `DATA_PATH`?
3. [ ] Wyniki zapisane do właściwej zmiennej/pliku?
4. [ ] Brak `input()` ani żadnych interaktywnych poleceń?
5. [ ] Brak odwołań do internetu (`requests`, `urllib`)?
6. [ ] Wyniki mają prawidłowy format (numpy array, lista, etc.)?
7. [ ] `torch.no_grad()` w ewaluacji?
8. [ ] Kernel Restart → Run All → działa?
9. [ ] Żadnych niezainicjalizowanych zmiennych?
10. [ ] Pamięć GPU nie jest wyczerpana?

### 🏆 Zasady punktacji
- Punkty za częściowe rozwiązania → **zawsze** próbuj
- Baseline jest lepszy niż zero → **zawsze** wyślij cokolwiek
- Metryki definiują wynik → sprawdź jakiej metryki wymaga zadanie
- Threshold jest kluczowy → sprawdź ile punktów potrzebujesz do awansu

In [ ]:
# === AUTOMATYCZNY SELF-CHECK ===

import sys
import importlib

def olympiad_self_check():
    """Uruchom przed oddaniem — sprawdza podstawowe wymagania."""
    print("🔍 OLYMPIAD SELF-CHECK")
    print("=" * 50)
    
    checks = []
    
    # 1. Sprawdź GPU
    gpu_ok = torch.cuda.is_available()
    checks.append(("GPU dostępne", gpu_ok))
    
    # 2. Sprawdź kluczowe biblioteki
    for lib in ['torch', 'numpy', 'pandas', 'sklearn', 'matplotlib']:
        try:
            importlib.import_module(lib)
            checks.append((f"Biblioteka {lib}", True))
        except ImportError:
            checks.append((f"Biblioteka {lib}", False))
    
    # 3. Sprawdź VRAM
    if gpu_ok:
        free_mem = (torch.cuda.get_device_properties(0).total_mem - 
                    torch.cuda.memory_allocated()) / 1e9
        checks.append((f"VRAM wolne ({free_mem:.1f} GB)", free_mem > 2.0))
    
    # 4. Wyświetl wyniki
    all_ok = True
    for check_name, status in checks:
        icon = "✅" if status else "❌"
        print(f"  {icon} {check_name}")
        if not status:
            all_ok = False
    
    print("\n" + ("🎉 Wszystko OK!" if all_ok else "⚠️ Napraw problemy przed oddaniem!"))
    return all_ok

olympiad_self_check()

print("\n" + "=" * 50)
print("\n🎯 PODSUMOWANIE STRATEGII:")
print("1. Triage → łatwe pierwsze, trudne na końcu")
print("2. FINAL_EVALUATION_MODE → testuj OBA tryby")
print("3. Bielik → tylko do NLP, potem zwolnij GPU")
print("4. Baseline → zawsze oddaj cokolwiek")
print("5. Debug → debug_tensor() + sprawdź wymiary")
print("\n💪 Powodzenia na 2. etapie III Olimpiady AI!")